In [1]:
# ===== ПОДРАЗДЕЛ 2.3 — МЕТОД ПРОСТОЙ ИТЕРАЦИИ. Инструменты =====
# Уравнение приводится к виду x = phi(x); строится x[n+1] = phi(x[n]).
# Достаточное условие сходимости:  |phi'(x)| <= q < 1  на отрезке [a; b].
#   0 < phi' < 1   — приближения идут к корню С ОДНОЙ СТОРОНЫ ("лестница")
#  -1 < phi' < 0   — приближения ПРЫГАЮТ вокруг корня ("спираль")

import math


def cobweb(phi, a, b, x0, steps=8, w=64, h=25, title=""):
    """Паутинная диаграмма: '/' — биссектриса y=x, '*' — y=phi(x), 'o' — итерации."""
    xs = [a + (b - a) * i / (w - 1) for i in range(w)]
    ys = [phi(x) for x in xs]
    lo, hi = min(min(ys), a), max(max(ys), b)
    pad = (hi - lo) * 0.05
    lo, hi = lo - pad, hi + pad

    col = lambda x: int(round((x - a) / (b - a) * (w - 1)))
    row = lambda y: int(round((hi - y) / (hi - lo) * (h - 1)))
    grid = [[" "] * w for _ in range(h)]

    for c, x in enumerate(xs):                       # биссектриса y = x
        r = row(x)
        if 0 <= r < h:
            grid[r][c] = "/"
    for c, y in enumerate(ys):                       # кривая y = phi(x)
        r = row(y)
        if 0 <= r < h:
            grid[r][c] = "*"

    x = x0                                           # траектория
    track = []
    for _ in range(steps):
        y = phi(x)
        track.append((x, y))
        r, c = row(y), col(x)
        if 0 <= r < h and 0 <= c < w:
            grid[r][c] = "o"
        r, c = row(y), col(y)
        if 0 <= r < h and 0 <= c < w:
            grid[r][c] = "o"
        x = y

    if title:
        print(title)
    for r in range(h):
        print(f"{hi - (hi - lo) * r / (h - 1):+8.4f} |" + "".join(grid[r]))
    print(" " * 9 + "+" + "-" * w)
    print(" " * 10 + f"{a:<{w // 2}.4g}{b:>{w // 2}.4g}")
    return track


def iterate(phi, x0, eps, cap=200, show=True, name=""):
    """Простая итерация с печатью расчётной таблицы."""
    if show:
        print(name)
        print(f"{'n':>3}{'x[n]':>16}{'phi(x[n])':>16}{'|x[n+1]-x[n]|':>17}{'знак разн.':>12}")
    x = x0
    for n in range(cap):
        y = phi(x)
        d = y - x
        if show:
            print(f"{n:>3}{x:>16.9f}{y:>16.9f}{abs(d):>17.3e}{'+' if d > 0 else '-':>12}")
        if abs(d) < eps:
            if show:
                print(f"   критерий |x[n+1]-x[n]| < {eps:g} выполнен на шаге {n}")
            return y, n
        x = y
    raise RuntimeError("не сошлось")


print("Инструменты готовы: cobweb (паутинная диаграмма), iterate (расчётная таблица).")


Инструменты готовы: cobweb (паутинная диаграмма), iterate (расчётная таблица).


In [2]:
# ===== УПРАЖНЕНИЕ 2.3, п. 1 =====
# Проиллюстрировать графически сходящийся итерационный процесс
# при условии -1 < phi'(x) < 0 на отрезке [a; b].

print("=" * 78)
print("УПРАЖНЕНИЕ 2.3, п.1  —  ГРАФИЧЕСКАЯ ИЛЛЮСТРАЦИЯ")
print("=" * 78)

print("""
Построение на рисунке: из точки x[n] на оси идём ВВЕРХ до кривой y = phi(x),
получаем y = phi(x[n]); затем ГОРИЗОНТАЛЬНО до биссектрисы y = x — так значение
переносится на ось абсцисс и становится следующим x[n+1]. Повторяем.

Знак производной решает, как выглядит ломаная:
""")

print("-" * 78)
print("СЛУЧАЙ A:   0 < phi'(x) < 1     (для сравнения)")
print("-" * 78)
phiA = lambda x: 0.5 * x + 0.7          # phi' = +0.5, корень x* = 1.4
trA = cobweb(phiA, 0.2, 2.6, 0.3, steps=7,
             title="\nphi(x) = 0,5x + 0,7      корень x* = 1,4")
print("\nпоследовательность:", "  ".join(f"{p[0]:.4f}" for p in trA))
print("Все приближения лежат ПО ОДНУ СТОРОНУ от корня и монотонно к нему")
print("подтягиваются — ломаная имеет вид ЛЕСТНИЦЫ.")

print("\n" + "-" * 78)
print("СЛУЧАЙ B:  -1 < phi'(x) < 0     <-- условие задачи, рис. 2.7")
print("-" * 78)
phiB = lambda x: -0.5 * x + 2.0         # phi' = -0.5, корень x* = 4/3
trB = cobweb(phiB, 0.2, 2.6, 0.3, steps=7,
             title="\nphi(x) = -0,5x + 2       корень x* = 1,3333")
print("\nпоследовательность:", "  ".join(f"{p[0]:.4f}" for p in trB))
print("""
Приближения ПЕРЕСКАКИВАЮТ через корень: одно левее, следующее правее.
Ломаная закручивается внутрь — вид СПИРАЛИ (паутины).

Практическое следствие этого случая, ради которого он и разбирается:
корень всегда заключён МЕЖДУ двумя соседними приближениями, поэтому
   |x[n+1] - x*|  <  |x[n+1] - x[n]|
и разность соседних приближений сама по себе служит надёжной оценкой
погрешности. При 0 < phi' < 1 это неверно — там приближения подходят
с одной стороны и разность занижает ошибку.
""")

print("-" * 78)
print("ПОЧЕМУ ВАЖНО |phi'| < 1:   тот же процесс при phi' = -1,5")
print("-" * 78)
phiC = lambda x: -1.5 * x + 3.5         # корень x* = 1.4, но |phi'| > 1
cobweb(phiC, -1.0, 3.5, 1.2, steps=5,
       title="\nphi(x) = -1,5x + 3,5     корень x* = 1,4 (процесс РАСХОДИТСЯ)")
x = 1.2
seq = []
for _ in range(6):
    seq.append(x)
    x = phiC(x)
print("\nпоследовательность:", "  ".join(f"{v:.4f}" for v in seq))
print("Спираль раскручивается НАРУЖУ: каждый скачок через корень длиннее предыдущего.")


УПРАЖНЕНИЕ 2.3, п.1  —  ГРАФИЧЕСКАЯ ИЛЛЮСТРАЦИЯ

Построение на рисунке: из точки x[n] на оси идём ВВЕРХ до кривой y = phi(x),
получаем y = phi(x[n]); затем ГОРИЗОНТАЛЬНО до биссектрисы y = x — так значение
переносится на ось абсцисс и становится следующим x[n+1]. Повторяем.

Знак производной решает, как выглядит ломаная:

------------------------------------------------------------------------------
СЛУЧАЙ A:   0 < phi'(x) < 1     (для сравнения)
------------------------------------------------------------------------------

phi(x) = 0,5x + 0,7      корень x* = 1,4
 +2.7200 |                                                                
 +2.6100 |                                                              //
 +2.5000 |                                                           ///  
 +2.3900 |                                                         //     
 +2.2800 |                                                      ///       
 +2.1700 |                                           

In [3]:
# ===== УПРАЖНЕНИЕ 2.3, п. 2 =====
# Уточнить наименьший по модулю отличный от нуля корень уравнения
# x*sin x - 1 = 0 методом простой итерации с точностью до 1e-5.

EPS = 1e-5
A, B = 1.05, 1.20               # отрезок отделения (из упражнения 2.1)

print("=" * 78)
print("УПРАЖНЕНИЕ 2.3, п.2  —  МЕТОД ПРОСТОЙ ИТЕРАЦИИ")
print("=" * 78)

f = lambda x: x * math.sin(x) - 1

print("""
ПРИВЕДЕНИЕ К ВИДУ x = phi(x)
Способ выбирается не произвольно: от него зависит, сойдётся ли процесс.
Разделив x*sin x = 1 на sin x, получаем phi(x) = 1/sin x.
""")

phi = lambda x: 1 / math.sin(x)
dphi = lambda x: -math.cos(x) / math.sin(x) ** 2

print("Проверка условия сходимости на отрезке [%.2f; %.2f]:" % (A, B))
print(f"{'x':>8}{'phi(x)':>14}{'phi_(x)':>14}")
vals = []
for i in range(7):
    x = A + (B - A) * i / 6
    vals.append(dphi(x))
    print(f"{x:>8.4f}{phi(x):>14.6f}{dphi(x):>14.6f}")
q = max(abs(v) for v in vals)
print(f"\n   phi' лежит в пределах ({min(vals):.4f}; {max(vals):.4f})")
print(f"   то есть -1 < phi'(x) < 0  —  это ровно СЛУЧАЙ B из пункта 1:")
print(f"   сходимость есть, приближения будут прыгать вокруг корня.")
print(f"   q = max|phi'| = {q:.4f} < 1")
print(f"   оценка числа шагов: ln(eps/|b-a|)/ln(q) = "
      f"{math.log(EPS / (B - A)) / math.log(q):.1f}  ->  около "
      f"{math.ceil(math.log(EPS / (B - A)) / math.log(q))} итераций")

print("\n" + "-" * 78)
print("ВАРИАНТ б) — ПРОГРАММНЫЙ РАСЧЁТ")
print("-" * 78)
x_it, n_it = iterate(phi, 1.15, EPS, name="\nРасчётная таблица, x0 = 1,15:")

x_ref, _ = iterate(phi, 1.15, 1e-13, show=False)
print(f"\n   найдено:              x = {x_it:.9f}")
print(f"   уточнённое значение:  x = {x_ref:.10f}")
print(f"   фактическая ошибка:   {abs(x_it - x_ref):.2e}   (требовалось < {EPS:g})")
print(f"   невязка:              f(x) = {f(x_it):+.3e}")
print(f"\n   ОТВЕТ: x = {x_it:.5f}   (и симметричный ему x = {-x_it:.5f})")

print("\n" + "-" * 78)
print("ВАРИАНТ а) — РУЧНАЯ РАСЧЁТНАЯ ТАБЛИЦА (первые шесть строк)")
print("-" * 78)
print(f"{'n':>3}{'x[n]':>12}{'sin x[n]':>12}{'1/sin x[n]':>14}{'разность':>13}")
x = 1.15
for n in range(6):
    s = math.sin(x)
    y = 1 / s
    print(f"{n:>3}{x:>12.6f}{s:>12.6f}{y:>14.6f}{y - x:>+13.6f}")
    x = y
print("  ... столбец разностей меняет знак на каждом шаге — та самая спираль")

print("\n" + "-" * 78)
print("НЕУДАЧНОЕ ПРИВЕДЕНИЕ: почему выбор phi(x) не свободен")
print("-" * 78)
psi = lambda x: math.asin(1 / x)
dpsi = lambda x: -1 / (x * math.sqrt(x * x - 1))
print("\nИз x*sin x = 1 можно выразить и иначе: sin x = 1/x, x = arcsin(1/x).")
print(f"{'x':>8}{'psi_(x)':>14}")
for i in range(4):
    x = A + (B - A) * i / 3
    print(f"{x:>8.4f}{dpsi(x):>14.6f}")
print("\n|psi'| > 1 на всём отрезке — условие сходимости нарушено. Проверим:")
x = 1.15
for n in range(6):
    try:
        y = psi(x)
    except ValueError:
        print(f"  шаг {n}: аргумент вышел из области определения — процесс разрушился")
        break
    print(f"  шаг {n}: x = {x:.6f}  ->  {y:.6f}")
    x = y
print("\nОдно и то же уравнение, два законных преобразования — и разные исходы.")
print("Проверка |phi'| < 1 на отрезке обязательна ДО начала счёта.")


УПРАЖНЕНИЕ 2.3, п.2  —  МЕТОД ПРОСТОЙ ИТЕРАЦИИ

ПРИВЕДЕНИЕ К ВИДУ x = phi(x)
Способ выбирается не произвольно: от него зависит, сойдётся ли процесс.
Разделив x*sin x = 1 на sin x, получаем phi(x) = 1/sin x.

Проверка условия сходимости на отрезке [1.05; 1.20]:
       x        phi(x)       phi_(x)
  1.0500      1.152840     -0.661292
  1.0750      1.136893     -0.614896
  1.1000      1.122073     -0.571100
  1.1250      1.108319     -0.529644
  1.1500      1.095574     -0.490300
  1.1750      1.083788     -0.452857
  1.2000      1.072916     -0.417128

   phi' лежит в пределах (-0.6613; -0.4171)
   то есть -1 < phi'(x) < 0  —  это ровно СЛУЧАЙ B из пункта 1:
   сходимость есть, приближения будут прыгать вокруг корня.
   q = max|phi'| = 0.6613 < 1
   оценка числа шагов: ln(eps/|b-a|)/ln(q) = 23.3  ->  около 24 итераций

------------------------------------------------------------------------------
ВАРИАНТ б) — ПРОГРАММНЫЙ РАСЧЁТ
-----------------------------------------------------------